In [12]:
import time
import requests
from collections import deque
from river import compose, preprocessing, linear_model,multiclass, metrics

In [2]:
API_KEY = "d6l9gmpr01qr0gn6cle0d6l9gmpr01qr0gn6cleg"
SYMBOL = "AAPL"

In [3]:
quote_url = "https://finnhub.io/api/v1/quote"

In [4]:
prices = deque(maxlen=30)
volumes = deque(maxlen=30)

In [13]:
model = compose.Pipeline(
    preprocessing.StandardScaler(),
    multiclass.OneVsRestClassifier(
        linear_model.LogisticRegression()
    )
)


In [15]:
metric = metrics.MacroF1()

last_prediction = None
last_features = None


In [7]:
def get_quote(symbol: str) -> dict:
    resp = requests.get(
        quote_url,
        params={"symbol": symbol, "token": API_KEY},
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()

In [8]:
def build_features(current_price: float) -> dict | None:
    if len(prices) < 6:
        return None

    ret_1 = (prices[-1] - prices[-2]) / prices[-2] if prices[-2] else 0.0
    ret_5 = (prices[-1] - prices[-6]) / prices[-6] if prices[-6] else 0.0

    ma_5 = sum(list(prices)[-5:]) / 5
    vol_5 = 0.0
    recent = list(prices)[-5:]
    mean_recent = sum(recent) / len(recent)
    vol_5 = (sum((p - mean_recent) ** 2 for p in recent) / len(recent)) ** 0.5

    x = {
        "ret_1": ret_1,
        "ret_5": ret_5,
        "dist_ma_5": (current_price - ma_5) / ma_5 if ma_5 else 0.0,
        "vol_5": vol_5,
    }
    return x


In [16]:
def predict_tag(x: dict) -> str:
    pred = model.predict_one(x)
    return pred if pred is not None else "neutral"

In [10]:
def get_user_feedback(predicted_tag: str) -> str:
    """
    Replace this with Streamlit/Flask/UI feedback.
    Example accepted tags: bullish, bearish, neutral
    """
    print(f"Predicted tag: {predicted_tag}")
    user_input = input("Enter true tag (bullish/bearish/neutral) or blank to skip: ").strip()
    return user_input

In [17]:
while True:
    try:
        q = get_quote(SYMBOL)
        current_price = q["c"]
        prices.append(current_price)

        x = build_features(current_price)
        if x is None:
            print("Collecting warm-up data...")
            time.sleep(5)
            continue

        predicted_tag = predict_tag(x)
        print(f"[{SYMBOL}] features={x} predicted={predicted_tag}")

        feedback = get_user_feedback(predicted_tag)

        if feedback in {"bullish", "bearish", "neutral"}:
            y_pred = model.predict_one(x)
            if y_pred is not None:
                metric.update(feedback, y_pred)

            model.learn_one(x, feedback)
            print("Model updated instantly from feedback.")
            print("Live metric:", metric)
        else:
            print("No valid feedback received; skipping update.")

        time.sleep(5)

    except KeyboardInterrupt:
        break
    except Exception as e:
        print("Error:", e)
        time.sleep(10)

[AAPL] features={'ret_1': 0.0, 'ret_5': 0.0, 'dist_ma_5': 0.0, 'vol_5': 0.0} predicted=bearish
Predicted tag: bearish
